# Stratified Merge — Khilgaon + Mohakhali

Merge two Roboflow datasets while preserving **every stratification variable**:

| Variable | How it's preserved |
|---|---|
| **Geographic** | `kh_` / `mo_` filename prefix + `location` column in manifest CSV |
| **Split** | train→train, val→val, test→test (never mixed) |
| **Vehicle class** | both `data.yaml` files remapped to one unified master class list |
| **Rare class** | `easybike` & `boat` dropped, `leguna` kept (Khilgaon-only) |

**Run order:** edit Config → run cells top to bottom → inspect summary → set `UPLOAD_BACK=True` and run the last cell.

## 1. Install & import

In [ ]:
!pip install --upgrade roboflow

In [2]:
import os
import shutil
import csv
from collections import defaultdict

import yaml
from roboflow import Roboflow

## 2. Configuration

Edit these values. Get your API key from **app.roboflow.com → Settings → API Keys**.

In [ ]:
API_KEY   = "YOUR_ROBOFLOW_API_KEY"
WORKSPACE = "YOUR_ROBOFLOW_WORKSPACE_NAME"

# (project_slug, version_number, location_tag)
KHILGAON  = ("khilgaon-augmented",1, "kh")
MOHAKHALI = ("mohakhali-augmentated",1, "mo")

EXPORT_FORMAT = "yolov11"


In [ ]:
# Classes to DROP entirely (too rare to be meaningful)
DROP_CLASSES = {"easybike", "boat"}

# Fixed master class order for the merged dataset (reproducible label indices)
MASTER_CLASSES = [
    "rickshaw", "car", "bike", "bus", "cng",
    "bicycle", "truck", "van", "leguna",
]

# Keep an image whose labels become empty after dropping a class?
KEEP_EMPTY_AFTER_DROP = False

MERGED_DIR   = "merged_dataset"
MANIFEST_CSV = "merged_manifest.csv"

UPLOAD_BACK         = False   # keep False for a dry run first
MERGED_PROJECT_SLUG = "khilgaon-mohakhali-merged-3"

SPLITS = ["train", "valid", "test"]   

master_name_to_idx = {name: i for i, name in enumerate(MASTER_CLASSES)}
rf = Roboflow(api_key=API_KEY)
print("Configured. Master classes:", MASTER_CLASSES)

## 3. Download both datasets

In [ ]:

def download(project_slug, version, fmt):
    print(f"Downloading {project_slug} v{version} ...")
    project = rf.workspace(WORKSPACE).project(project_slug)
    dataset = project.version(version).download(fmt)
    print("  ->", dataset.location)
    return dataset.location

kh_path = download(KHILGAON[0],  KHILGAON[1],  EXPORT_FORMAT)
mo_path = download(MOHAKHALI[0], MOHAKHALI[1], EXPORT_FORMAT)

In [ ]:
!rm -rf khilgaon-augmented-1

## 4. Read class maps (index → name) from each `data.yaml`

In [ ]:
def read_class_map(dataset_path):
    with open(os.path.join(dataset_path, "data.yaml")) as f:
        data = yaml.safe_load(f)
    names = data["names"]
    if isinstance(names, dict):
        return {int(k): v for k, v in names.items()}
    return {i: n for i, n in enumerate(names)}

kh_idx_to_name = read_class_map(kh_path)
mo_idx_to_name = read_class_map(mo_path)

print("Khilgaon :", list(kh_idx_to_name.values()))
print("Mohakhali:", list(mo_idx_to_name.values()))

## 5. Helper functions — prepare folders, locate splits, remap & copy

In [15]:
def prepare_dirs():
    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)
    for split in SPLITS:
        os.makedirs(os.path.join(MERGED_DIR, split, "images"), exist_ok=True)
        os.makedirs(os.path.join(MERGED_DIR, split, "labels"), exist_ok=True)

def find_split_dir(dataset_path, split):
    """YOLOv8 uses 'valid'; some exports use 'val'. Handle both."""
    candidates = [split] + (["val"] if split == "valid" else [])
    for c in candidates:
        p = os.path.join(dataset_path, c)
        if os.path.isdir(p):
            return p
    return None

In [16]:
def remap_and_copy(dataset_path, location_tag, idx_to_name, stats, manifest_rows):
    """Copy each image (location-prefixed) and rewrite its label file:
       remap class indices to the master list and drop unwanted classes."""
    for split in SPLITS:
        src_split = find_split_dir(dataset_path, split)
        if src_split is None:
            print(f"  [warn] {location_tag}: no '{split}' folder, skipping")
            continue

        img_dir = os.path.join(src_split, "images")
        lbl_dir = os.path.join(src_split, "labels")
        if not os.path.isdir(img_dir):
            continue

        for img_name in os.listdir(img_dir):
            base, ext = os.path.splitext(img_name)
            if ext.lower() not in (".jpg", ".jpeg", ".png", ".bmp"):
                continue

            src_lbl = os.path.join(lbl_dir, base + ".txt")
            new_lines = []
            if os.path.exists(src_lbl):
                with open(src_lbl) as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split()
                        cname = idx_to_name.get(int(parts[0]))
                        if cname is None:
                            continue
                        if cname in DROP_CLASSES:
                            stats[(location_tag, split, "DROPPED:" + cname)] += 1
                            continue
                        if cname not in master_name_to_idx:
                            continue
                        new_idx = master_name_to_idx[cname]
                        new_lines.append(" ".join([str(new_idx)] + parts[1:]))
                        stats[(location_tag, split, cname)] += 1

            if not new_lines and not KEEP_EMPTY_AFTER_DROP:
                stats[(location_tag, split, "SKIPPED_EMPTY")] += 1
                continue

            new_img_name = f"{location_tag}_{img_name}"
            new_base     = f"{location_tag}_{base}"
            shutil.copy2(os.path.join(img_dir, img_name),
                         os.path.join(MERGED_DIR, split, "images", new_img_name))
            with open(os.path.join(MERGED_DIR, split, "labels", new_base + ".txt"), "w") as f:
                f.write("\n".join(new_lines) + ("\n" if new_lines else ""))

            manifest_rows.append({
                "filename":    new_img_name,
                "location":    "khilgaon" if location_tag == "kh" else "mohakhali",
                "split":       split,
                "num_objects": len(new_lines),
            })

## 6. Run the merge (split-by-split, location-tagged)

In [ ]:
prepare_dirs()

stats = defaultdict(int)
manifest_rows = []

remap_and_copy(kh_path, KHILGAON[2],  kh_idx_to_name, stats, manifest_rows)
remap_and_copy(mo_path, MOHAKHALI[2], mo_idx_to_name, stats, manifest_rows)

print(f"Merged {len(manifest_rows)} images.")

## 7. Write merged `data.yaml` + manifest CSV

In [ ]:
# data.yaml
with open(os.path.join(MERGED_DIR, "data.yaml"), "w") as f:
    yaml.safe_dump({
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(MASTER_CLASSES),
        "names": MASTER_CLASSES,
    }, f, sort_keys=False)

# manifest (location stratification record)
with open(MANIFEST_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["filename", "location", "split", "num_objects"])
    w.writeheader()
    w.writerows(manifest_rows)

print("Wrote data.yaml and", MANIFEST_CSV)

## 8. Summary — verify the merge

In [ ]:
counts = defaultdict(int)
for row in manifest_rows:
    counts[(row["location"], row["split"])] += 1

print("Images per split:")
for split in SPLITS:
    kh = counts[("khilgaon", split)]
    mo = counts[("mohakhali", split)]
    print(f"  {split:6s}: khilgaon={kh:5d}  mohakhali={mo:5d}  merged={kh+mo:5d}")
print(f"  TOTAL : {len(manifest_rows)} images\n")

print("Object counts per class:")
classes = sorted({k[2] for k in stats if not k[2].startswith(('DROPPED','SKIPPED'))})
print(f"  {'class':12s} {'KH':>8s} {'MO':>8s} {'total':>8s}")
for c in classes:
    kh = sum(v for k, v in stats.items() if k[0]=='kh' and k[2]==c)
    mo = sum(v for k, v in stats.items() if k[0]=='mo' and k[2]==c)
    print(f"  {c:12s} {kh:8d} {mo:8d} {kh+mo:8d}")

print("\nDropped / skipped:")
for k, v in sorted(stats.items()):
    if k[2].startswith("DROPPED"):
        print(f"  {k[0]}/{k[1]}: {k[2]} -> {v} removed")
skipped = sum(v for k, v in stats.items() if k[2]=="SKIPPED_EMPTY")
if skipped:
    print(f"  images skipped (empty after drop): {skipped}")

## 9. Verify the 70 / 20 / 10 ratio held per class

Reads the merged labels back and checks each class lands within ±5% of target.

In [ ]:
def count_objects_in_split(split):
    per_class = defaultdict(int)
    lbl_dir = os.path.join(MERGED_DIR, split, "labels")
    if not os.path.isdir(lbl_dir):
        return per_class
    for fn in os.listdir(lbl_dir):
        if not fn.endswith(".txt"):
            continue
        with open(os.path.join(lbl_dir, fn)) as f:
            for line in f:
                line = line.strip()
                if line:
                    per_class[MASTER_CLASSES[int(line.split()[0])]] += 1
    return per_class

by_split = {s: count_objects_in_split(s) for s in SPLITS}

print(f"{'class':12s} {'train%':>8s} {'val%':>8s} {'test%':>8s}  status")
for c in MASTER_CLASSES:
    tr = by_split['train'].get(c, 0)
    va = by_split['valid'].get(c, 0)
    te = by_split['test'].get(c, 0)
    tot = tr + va + te
    if tot == 0:
        print(f"{c:12s} {'--':>8s} {'--':>8s} {'--':>8s}  (absent)")
        continue
    p_tr, p_va, p_te = tr/tot*100, va/tot*100, te/tot*100
    ok = abs(p_tr-70)<=5 and abs(p_va-20)<=5 and abs(p_te-10)<=5
    print(f"{c:12s} {p_tr:7.1f}% {p_va:7.1f}% {p_te:7.1f}%  {'OK' if ok else 'CHECK'}")

In [ ]:
# ---- Daylight dataset config — DIFFERENT WORKSPACE (edit all four values) ----
DAYLIGHT_API_KEY   = "PUT_DAYLIGHT_WORKSPACE_API_KEY_HERE"   # <-- key from the OTHER workspace
DAYLIGHT_WORKSPACE = "other-workspace-id"                    # <-- the other workspace slug
# (project_slug, version_number, location_tag)
DAYLIGHT           = ("daylight-augmented", 1, "dl")         # <-- change slug & version
DAYLIGHT_LOCATION  = "daylight"                              # value written to manifest 'location'

# Separate Roboflow client for the other workspace (does not touch the existing `rf`)
rf_dl = Roboflow(api_key=DAYLIGHT_API_KEY)

def download_from(rf_client, workspace, project_slug, version, fmt):
    print(f"Downloading {workspace}/{project_slug} v{version} ...")
    project = rf_client.workspace(workspace).project(project_slug)
    dataset = project.version(version).download(fmt)
    print("  ->", dataset.location)
    return dataset.location

dl_path = download_from(rf_dl, DAYLIGHT_WORKSPACE, DAYLIGHT[0], DAYLIGHT[1], EXPORT_FORMAT)
print("Daylight downloaded to:", dl_path)


In [ ]:
# ---- Daylight dataset config — DIFFERENT WORKSPACE (edit all four values) ----
DAYLIGHT_API_KEY   = "EW2VqJI42QuaVSgqEEEz"   # <-- key from the OTHER workspace
DAYLIGHT_WORKSPACE = "towfiqurs-workspace"                    # <-- the other workspace slug
# (project_slug, version_number, location_tag)
DAYLIGHT           = ("vehicle-detection-in-bangladesh-6syeh", 1, "dl")         # <-- change slug & version
DAYLIGHT_LOCATION  = "daylight"                              # value written to manifest 'location'

# Separate Roboflow client for the other workspace (does not touch the existing `rf`)
rf_dl = Roboflow(api_key=DAYLIGHT_API_KEY)

def download_from(rf_client, workspace, project_slug, version, fmt):
    print(f"Downloading {workspace}/{project_slug} v{version} ...")
    project = rf_client.workspace(workspace).project(project_slug)
    dataset = project.version(version).download(fmt)
    print("  ->", dataset.location)
    return dataset.location

dl_path = download_from(rf_dl, DAYLIGHT_WORKSPACE, DAYLIGHT[0], DAYLIGHT[1], EXPORT_FORMAT)
print("Daylight downloaded to:", dl_path)


In [ ]:
dl_idx_to_name = read_class_map(dl_path)
print("Daylight classes:", list(dl_idx_to_name.values()))

# Sanity check vs the master class list
unknown = {n for n in dl_idx_to_name.values()
           if n not in master_name_to_idx and n not in DROP_CLASSES}
print("Will DROP (in DROP_CLASSES):", sorted(c for c in dl_idx_to_name.values() if c in DROP_CLASSES) or "none")
print("NOT in master list -> skipped:", sorted(unknown) or "none")


In [13]:
def remap_and_copy_loc(dataset_path, location_tag, location_name, idx_to_name, stats, manifest_rows):
    """Like remap_and_copy(), but writes an explicit `location_name` to the manifest
    (the original hard-codes only kh/mo). Used for any extra source, e.g. daylight."""
    for split in SPLITS:
        src_split = find_split_dir(dataset_path, split)
        if src_split is None:
            print(f"  [warn] {location_tag}: no '{split}' folder, skipping")
            continue

        img_dir = os.path.join(src_split, "images")
        lbl_dir = os.path.join(src_split, "labels")
        if not os.path.isdir(img_dir):
            continue

        for img_name in os.listdir(img_dir):
            base, ext = os.path.splitext(img_name)
            if ext.lower() not in (".jpg", ".jpeg", ".png", ".bmp"):
                continue

            src_lbl = os.path.join(lbl_dir, base + ".txt")
            new_lines = []
            if os.path.exists(src_lbl):
                with open(src_lbl) as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split()
                        cname = idx_to_name.get(int(parts[0]))
                        if cname is None:
                            continue
                        if cname in DROP_CLASSES:
                            stats[(location_tag, split, "DROPPED:" + cname)] += 1
                            continue
                        if cname not in master_name_to_idx:
                            continue
                        new_idx = master_name_to_idx[cname]
                        new_lines.append(" ".join([str(new_idx)] + parts[1:]))
                        stats[(location_tag, split, cname)] += 1

            if not new_lines and not KEEP_EMPTY_AFTER_DROP:
                stats[(location_tag, split, "SKIPPED_EMPTY")] += 1
                continue

            new_img_name = f"{location_tag}_{img_name}"
            new_base     = f"{location_tag}_{base}"
            shutil.copy2(os.path.join(img_dir, img_name),
                         os.path.join(MERGED_DIR, split, "images", new_img_name))
            with open(os.path.join(MERGED_DIR, split, "labels", new_base + ".txt"), "w") as f:
                f.write("\n".join(new_lines) + ("\n" if new_lines else ""))

            manifest_rows.append({
                "filename":    new_img_name,
                "location":    location_name,
                "split":       split,
                "num_objects": len(new_lines),
            })


In [ ]:
# Append daylight into the EXISTING merged_dataset.
# IMPORTANT: do NOT call prepare_dirs() here — that wipes khilgaon+mohakhali.
# This cell is idempotent: it first removes any prior daylight ('dl_') entries,
# so you can safely re-run it.

dl_tag = DAYLIGHT[2]

# 1) strip any previously-added daylight files + manifest rows + stats
for split in SPLITS:
    for sub in ("images", "labels"):
        d = os.path.join(MERGED_DIR, split, sub)
        if os.path.isdir(d):
            for fn in list(os.listdir(d)):
                if fn.startswith(f"{dl_tag}_"):
                    os.remove(os.path.join(d, fn))
manifest_rows = [r for r in manifest_rows if not r["filename"].startswith(f"{dl_tag}_")]
for k in [k for k in list(stats) if k[0] == dl_tag]:
    del stats[k]

# 2) remap + copy daylight (split preserved, same drop/master-class rules)
before = len(manifest_rows)
remap_and_copy_loc(dl_path, dl_tag, DAYLIGHT_LOCATION, dl_idx_to_name, stats, manifest_rows)
print(f"Added {len(manifest_rows) - before} daylight images. Grand total: {len(manifest_rows)}")


In [ ]:
# Re-write data.yaml (class list unchanged) and refresh the manifest so it now
# records all three sources (khilgaon + mohakhali + daylight).
with open(os.path.join(MERGED_DIR, "data.yaml"), "w") as f:
    yaml.safe_dump({
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(MASTER_CLASSES),
        "names": MASTER_CLASSES,
    }, f, sort_keys=False)

with open(MANIFEST_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["filename", "location", "split", "num_objects"])
    w.writeheader()
    w.writerows(manifest_rows)

print("Updated data.yaml and", MANIFEST_CSV, "— daylight now included.")


In [ ]:
counts = defaultdict(int)
for row in manifest_rows:
    counts[(row["location"], row["split"])] += 1

locations = ["khilgaon", "mohakhali", DAYLIGHT_LOCATION]
print("Images per split (3 sources):")
for split in SPLITS:
    vals = {loc: counts[(loc, split)] for loc in locations}
    line = "  ".join(f"{loc}={vals[loc]:5d}" for loc in locations)
    print(f"  {split:6s}: {line}  merged={sum(vals.values()):5d}")
print(f"  TOTAL : {len(manifest_rows)} images\n")

print("Object counts per class (incl. daylight):")
classes = sorted({k[2] for k in stats if not k[2].startswith(('DROPPED', 'SKIPPED'))})
tags = [KHILGAON[2], MOHAKHALI[2], DAYLIGHT[2]]
print(f"  {'class':12s} {'KH':>8s} {'MO':>8s} {'DL':>8s} {'total':>8s}")
for c in classes:
    row = {t: sum(v for k, v in stats.items() if k[0] == t and k[2] == c) for t in tags}
    print(f"  {c:12s} {row[KHILGAON[2]]:8d} {row[MOHAKHALI[2]]:8d} {row[DAYLIGHT[2]]:8d} {sum(row.values()):8d}")

print("\nDaylight dropped / skipped:")
for k, v in sorted(stats.items()):
    if k[0] == DAYLIGHT[2] and k[2].startswith("DROPPED"):
        print(f"  {k[0]}/{k[1]}: {k[2]} -> {v} removed")
dl_skipped = sum(v for k, v in stats.items() if k[0] == DAYLIGHT[2] and k[2] == "SKIPPED_EMPTY")
if dl_skipped:
    print(f"  daylight images skipped (empty after drop): {dl_skipped}")
